# Título: Fine-tuning SBERT para ranking de CVs y ofertas laborales
### Descripción: Entrenamiento supervisado con pares (cv_text, oferta_text, label)
### Autor: Pablo Téllez López
### Fecha: 24-06-2025

## 1. Setup inicial

In [1]:
# Montar Google Drive si usas DATA desde allí
from google.colab import drive
drive.mount('/content/drive',
#            force_remount=True
            )

# Instalar librerías necesarias
#!pip install -U sentence-transformers datasets
#!pip install scikit-learn pandas tqdm


Mounted at /content/drive


## 2. Carga y preparación de los datos

In [2]:
%cd /content/drive/MyDrive/TFM/

# Importar librerías
import pandas as pd
from sentence_transformers import InputExample
from sklearn.model_selection import train_test_split

# Cargar archivos de entrenamiento y validación desde interim/
train_df = pd.read_csv('interim/pairs_train_clean.csv')
val_df = pd.read_csv('interim/pairs_val_clean.csv')

# Inspección rápida (opcional)
print("Train size:", train_df.shape)
print("Val size:", val_df.shape)
print(train_df.columns)

# Crear InputExamples para SBERT
train_examples = [
    InputExample(texts=[row['cv_text'], row['offer_text']], label=float(row['label']))
    for _, row in train_df.iterrows()
]

/content/drive/MyDrive/TFM
Train size: (1354, 7)
Val size: (285, 7)
Index(['cv_id', 'cv_text', 'cv_category', 'offer_id', 'offer_text',
       'offer_category', 'label'],
      dtype='object')


In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3. Definición del modelo y pérdida



In [4]:
from sentence_transformers import losses
train_loss = losses.CosineSimilarityLoss(model)


## 4. Dataloader y Evaluador

In [5]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)


In [6]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

sentences1 = val_df['cv_text'].tolist()
sentences2 = val_df['offer_text'].tolist()
scores = val_df['label'].astype(float).tolist()

evaluator = EmbeddingSimilarityEvaluator(sentences1, sentences2, scores)


## 5. Entrenamiento y Guardado

In [7]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    evaluation_steps=500,
    epochs=4,
    warmup_steps=100,
    output_path='sbert/fine_tuned_model',
    show_progress_bar=True
)


/usr/local/lib/python3.11/dist-packages/datasets/table.py:1395: FutureWarning: promote has been superseded by promote_options='default'.
  block_group = [InMemoryTable(cls._concat_blocks(list(block_group), axis=axis))]
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ptellezlopez (ptellezlopez-universidad-loyola) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
85,No log,No log,0.730159,0.621115
170,No log,No log,0.742633,0.712470
255,No log,No log,0.727221,0.712274
340,No log,No log,0.728804,0.713642


In [8]:
model.save('sbert/fine_tuned_model')  # Solo si no se guardó antes

## 6. Evaluación final y Generación de rankings

In [9]:
# Cargar modelo entrenado (si no está ya cargado)
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('sbert/fine_tuned_model')

# Cargar dataset de test
test_df = pd.read_csv("interim/dataset_ranking_test.csv")

# Obtener embeddings
cv_embeddings = model.encode(test_df["texto_cv"].tolist(), batch_size=64, convert_to_numpy=True)
offer_embeddings = model.encode(test_df["texto_oferta"].tolist(), batch_size=64, convert_to_numpy=True)

# Calcular similitud de coseno para cada par
scores = cosine_similarity(cv_embeddings, offer_embeddings).diagonal()
test_df["score"] = scores

# Recalcular el ranking (puedes sobreescribir o crear una nueva columna si prefieres)
test_df["rank"] = test_df.groupby("cv_id")["score"].rank(method="first", ascending=False)

# Guardar con todas las columnas originales más score actualizado
test_df.to_csv("processed/rankings_test_sbert.csv", index=False)


In [10]:
display(test_df)

,cv_id,offer_id,sector_oferta,score,label_binario,rank,texto_cv,texto_oferta
0,cv00.pdf,cdatos_0,cdatos,0.894083,1,5.0,EXPERIENCIA LABORAL Analista de Datos Junior A...,los ingenieros de software junior son profesio...
1,cv00.pdf,cdatos_1,cdatos,0.938794,1,3.0,EXPERIENCIA LABORAL Analista de Datos Junior A...,🚀 unete a inetum como responsable de data & ai...
2,cv00.pdf,cdatos_4,cdatos,0.946748,1,2.0,EXPERIENCIA LABORAL Analista de Datos Junior A...,✨ 💡 ¡buscamos un/a consultor/a de business int...
3,cv00.pdf,cdatos_3,cdatos,0.938001,1,4.0,EXPERIENCIA LABORAL Analista de Datos Junior A...,🚀 ¡seguimos buscando talento…y nos encantaria ...
4,cv00.pdf,cdatos_2,cdatos,0.949287,1,1.0,EXPERIENCIA LABORAL Analista de Datos Junior A...,estamos ampliando nuestro equipo de ciencia de...
...,...,...,...,...,...,...,...,...
242,cv12.pdf,jurista_2,jurista,0.059759,0,10.0,Carmen Linares Vazquez Perfil Mi formacion en ...,"¡es tu oportunidad! ecointegral ingenieria, in..."
243,cv12.pdf,jurista_1,jurista,0.052469,0,12.0,Carmen Linares Vazquez Perfil Mi formacion en ...,en ey tendras la oportunidad de construir una ...
244,cv12.pdf,jurista_0,jurista,0.030232,0,15.0,Carmen Linares Vazquez Perfil Mi formacion en ...,"consejos: haz un resumen del puesto, explica q..."
245,cv12.pdf,jurista_4,jurista,0.057380,0,11.0,Carmen Linares Vazquez Perfil Mi formacion en ...,betancourt selecciona: abogado/a senior labora...
